# Digital Twins and Simulation

> Computational Analysis of Social Complexity
>
> Fall 2025, Spencer Lyon

**Prerequisites**

- Production Networks (Week 5)
- Agent-Based Models (Week 6)
- Network Traffic and Game Theory (Week 8)
- AI Agents and Swarms (L.A3.01)
- Strategic AI Agents (L.A3.02)

**Outcomes**

- Understand digital twin architecture and components
- Build predictive simulation environments with AI agents
- Calibrate agent models from real-world data patterns
- Design and test policy interventions in simulated systems
- Apply digital twins to economic and social systems

**References**

- [Digital Twins: State of the Art Theory and Practice](https://link.springer.com/article/10.1007/s00170-020-05742-6)
- [Agent-Based Models and Digital Twins](https://www.sciencedirect.com/science/article/pii/S0167739X21002831)
- [Urban Digital Twins](https://www.nature.com/articles/s42949-020-00002-y)
- QuantEcon Production Networks lecture

## Introduction: From Models to Mirrors

- Throughout this course we've built computational models of social systems
  - Network models capturing relationships and information flow
  - Agent-based models showing emergence from simple rules
  - Game-theoretic models of strategic interaction
  - Production networks modeling economic interdependencies
- These models help us *understand* complex systems
- But what if we could build models that don't just explain -- but *predict*?
- What if our models could sync with real data in real-time?
- What if we could test policy changes before implementing them in the real world?

This is the promise of **digital twins**.

### What is a Digital Twin?

- A **digital twin** is a virtual representation of a physical system that:
  1. **Mirrors** the current state through real-time data synchronization
  2. **Predicts** future states using simulation and modeling
  3. **Prescribes** optimal actions through what-if analysis
- Originally developed for manufacturing and aerospace (NASA used digital twins for Apollo missions!)
- Now applied to:
  - Cities and urban planning
  - Supply chains and logistics
  - Healthcare systems
  - Financial markets
  - Social networks

**Key Insight**: AI agents enable digital twins that are more realistic than traditional ABMs because they can:
- Learn from data rather than following fixed rules
- Adapt to changing conditions
- Model complex, nuanced human behavior

### Digital Twins vs Traditional Models

| Aspect | Traditional Model | Digital Twin |
|--------|------------------|-------------|
| Purpose | Understanding | Prediction + Prescription |
| Data Connection | One-time calibration | Continuous real-time sync |
| Agents | Fixed behavioral rules | AI-powered, learning agents |
| Validation | Historical fit | Ongoing accuracy tracking |
| Use Case | Research, education | Operational decision-making |

**Example: Traffic Modeling**
- Traditional ABM: Agents follow fixed routing rules based on distance
- Digital Twin: AI driver agents learn from real GPS data, adapt to conditions, predict congestion

## Digital Twin Architecture

A digital twin system has three main components:

1. **Physical System**: The real-world entity being modeled
2. **Digital Model**: Computational representation with AI agents
3. **Data Pipeline**: Bidirectional flow between physical and digital

Let's build this framework in Julia.

In [ ]:
using Agents
using Graphs, SimpleWeightedGraphs
using DataFrames
using Statistics, StatsBase
using Plots
using Distributions
using LinearAlgebra

### Component 1: Physical System Interface

- In a real deployment, this would connect to sensors, APIs, databases
- For our purposes, we'll simulate data streams that would come from a physical system
- The key is to define a clean interface for data ingestion

In [ ]:
"""
Abstract type for data sources that feed into digital twin
"""
abstract type DataSource end

"""
Simulated real-time data stream
In production, this would connect to actual sensors/APIs
"""
mutable struct SimulatedDataStream <: DataSource
    current_time::Int
    data_history::Vector{Dict{Symbol, Any}}
    noise_level::Float64
end

function SimulatedDataStream(noise_level=0.1)
    SimulatedDataStream(0, Dict{Symbol, Any}[], noise_level)
end

"""
Pull latest data from the source
"""
function get_latest_data(source::SimulatedDataStream)
    if isempty(source.data_history)
        return nothing
    end
    return source.data_history[end]
end

"""
Push new observation to data stream
"""
function push_data!(source::SimulatedDataStream, data::Dict)
    source.current_time += 1
    push!(source.data_history, merge(data, Dict(:timestamp => source.current_time)))
end

### Component 2: Digital Model Core

- This is where our agent-based simulation lives
- Unlike traditional ABMs, agents have AI components that learn from data
- The model maintains both current state and predictive capabilities

In [ ]:
"""
Core digital twin structure
"""
mutable struct DigitalTwin
    model::AgentBasedModel  # The simulation model
    data_source::DataSource  # Connection to physical system
    sync_interval::Int  # How often to sync with real data
    last_sync::Int  # Last synchronization time
    prediction_horizon::Int  # How far ahead to predict
    metrics::Dict{Symbol, Vector{Float64}}  # Performance tracking
end

"""
Synchronize digital twin with physical system data
"""
function sync!(twin::DigitalTwin)
    data = get_latest_data(twin.data_source)
    if data === nothing
        return
    end
    
    # Update model state based on real data
    # This is model-specific and would be implemented by subtype
    update_from_data!(twin.model, data)
    
    twin.last_sync = data[:timestamp]
end

"""
Run predictive simulation forward from current state
"""
function predict(twin::DigitalTwin, steps::Int)
    # Create a copy of the model to run prediction without affecting current state
    prediction_model = deepcopy(twin.model)
    
    # Run simulation forward
    predicted_states = []
    for _ in 1:steps
        step!(prediction_model)
        push!(predicted_states, extract_state(prediction_model))
    end
    
    return predicted_states
end

### Component 3: Calibration and Learning

- A key challenge: how do we ensure our digital twin stays accurate?
- **Calibration**: Adjusting model parameters to match observed data
- **Online learning**: Continuously updating as new data arrives

This is where AI agents shine -- they can learn behavioral patterns from data!

In [ ]:
"""
Measure prediction accuracy
"""
function compute_prediction_error(twin::DigitalTwin, actual_data::Dict, predicted_state::Dict)
    # Compare predicted vs actual on key metrics
    errors = Float64[]
    
    for key in keys(actual_data)
        if haskey(predicted_state, key)
            actual = actual_data[key]
            predicted = predicted_state[key]
            
            # Normalized error
            if actual != 0
                error = abs(predicted - actual) / abs(actual)
                push!(errors, error)
            end
        end
    end
    
    return isempty(errors) ? 0.0 : mean(errors)
end

"""
Calibrate model parameters based on recent prediction errors
"""
function calibrate!(twin::DigitalTwin, learning_rate=0.1)
    # In a full implementation, this would use optimization or ML techniques
    # to adjust agent behavioral parameters based on prediction errors
    
    if !haskey(twin.metrics, :prediction_error) || isempty(twin.metrics[:prediction_error])
        return
    end
    
    recent_error = twin.metrics[:prediction_error][end]
    
    # Adjust model parameters (simplified example)
    # In practice, this would use gradient descent, Bayesian updating, etc.
    adjust_parameters!(twin.model, recent_error, learning_rate)
end

## Social System Simulation

- Now let's apply digital twins to social systems
- Challenge: human behavior is complex, adaptive, and strategic
- Traditional ABMs use fixed rules: "if unhappy, move" (Schelling model)
- AI agents can learn more realistic behavioral patterns from data

We'll build a simple example: modeling information spread with learning agents.

### Example: Information Diffusion Digital Twin

- Physical system: Social network (Twitter, Facebook, etc.)
- Goal: Predict how information spreads
- Traditional approach: Fixed sharing probability
- Digital twin approach: Agents learn sharing behavior from engagement patterns

In [ ]:
@agent struct SocialAgent(GraphAgent)
    # Agent state
    has_information::Bool
    exposure_count::Int  # How many times seen the information
    
    # Learned behavior model (simplified)
    share_threshold::Float64  # Probability threshold for sharing
    engagement_history::Vector{Bool}  # Past sharing decisions
    
    # Agent characteristics
    influence::Float64  # How influential this agent is
end

function create_social_network_twin(n_agents=100, network_type=:scale_free)
    # Create network structure
    if network_type == :scale_free
        graph = barabasi_albert(n_agents, 3)
    else
        graph = erdos_renyi(n_agents, 0.05)
    end
    
    # Initialize model
    model = StandardABM(
        SocialAgent,
        graph;
        agent_step! = social_agent_step!,
        properties = Dict(
            :total_informed => 0,
            :new_informed => 0,
            :time_step => 0
        )
    )
    
    # Create agents with learned parameters
    for i in 1:n_agents
        influence = rand(Beta(2, 5))  # Most agents have low influence
        share_threshold = rand() * 0.5 + 0.3  # Start with reasonable defaults
        
        add_agent!(
            i,  # position in graph
            model,
            false,  # has_information
            0,  # exposure_count
            share_threshold,
            Bool[],  # engagement_history
            influence
        )
    end
    
    # Seed information with a few influential agents
    seed_ids = sample(1:n_agents, 5, replace=false)
    for id in seed_ids
        model[id].has_information = true
        model.total_informed += 1
    end
    
    return model
end

### Agent Decision-Making with Learning

- Unlike traditional ABMs where agents follow fixed rules
- These agents adjust their behavior based on past experience
- They learn what types of content to share based on engagement

In [ ]:
function social_agent_step!(agent::SocialAgent, model)
    # Check if neighbors have information
    neighbor_ids = nearby_ids(agent, model)
    informed_neighbors = sum([model[id].has_information for id in neighbor_ids])
    
    # If agent doesn't have info, check if they receive it
    if !agent.has_information && informed_neighbors > 0
        agent.exposure_count += 1
        
        # Probability of adopting increases with exposures
        adoption_prob = 1 - exp(-0.3 * agent.exposure_count)
        
        if rand() < adoption_prob
            agent.has_information = true
            model.total_informed += 1
            model.new_informed += 1
        end
    end
    
    # If agent has info, decide whether to share (learned behavior)
    if agent.has_information
        # Sharing probability depends on:
        # 1. Agent's learned threshold
        # 2. Agent's influence
        # 3. How many neighbors don't have it yet (novelty)
        
        uninformed_neighbors = length(neighbor_ids) - informed_neighbors
        novelty_factor = uninformed_neighbors / max(1, length(neighbor_ids))
        
        share_prob = agent.influence * novelty_factor
        
        will_share = rand() < share_prob && share_prob > agent.share_threshold
        push!(agent.engagement_history, will_share)
        
        # Learn: adjust threshold based on engagement success
        if length(agent.engagement_history) >= 10
            recent_engagement = mean(agent.engagement_history[end-9:end])
            # If sharing a lot, become more selective (increase threshold)
            # If sharing rarely, become less selective (decrease threshold)
            target_rate = 0.3  # Target sharing about 30% of the time
            agent.share_threshold += 0.01 * (recent_engagement - target_rate)
            agent.share_threshold = clamp(agent.share_threshold, 0.1, 0.9)
        end
    end
end

### Running the Social Digital Twin

Let's see how information spreads through our network with learning agents.

In [ ]:
# Create the model
social_model = create_social_network_twin(200, :scale_free)

# Collect data as we simulate
adata = [:has_information, :exposure_count, :share_threshold, :influence]
mdata = [:total_informed, :new_informed, :time_step]

# Run simulation
data, mdata_df = run!(social_model, 50; adata, mdata)

println("Simulation complete!")
println("Final informed: $(social_model.total_informed) / 200")

In [ ]:
# Visualize the diffusion process
plot(
    mdata_df.time,
    mdata_df.total_informed,
    xlabel="Time Step",
    ylabel="Number of Informed Agents",
    label="Total Informed",
    title="Information Diffusion with Learning Agents",
    legend=:bottomright,
    linewidth=2
)

### Exercise 1: Calibration from Real Data

Suppose we observe the following real data from a social network:
- Day 1: 5 people have the information
- Day 2: 23 people have the information  
- Day 3: 67 people have the information
- Day 4: 134 people have the information

**Tasks:**
1. Run the model for 4 steps and compare to observed data
2. Calculate the prediction error at each time step
3. Adjust the initial seeding (number of initially informed agents) to better match the data
4. Try different network structures (`:scale_free` vs `:random`) - which matches better?

In [ ]:
# TODO: Your calibration code here
observed_data = [5, 23, 67, 134]

# Create model and run for 4 steps
# Compare predictions to observed_data
# Compute error metrics

## Economic Digital Twins

- Recall from Week 5: production networks model economic interdependencies
- Input-output tables show how sectors depend on each other
- Digital twins can predict cascading effects of sector-specific shocks

Let's build a digital twin of a supply chain network.

### Supply Chain Digital Twin

- Physical system: Network of suppliers, manufacturers, distributors
- Each node: An AI agent representing a firm
- Edges: Supply relationships with lead times and capacities
- Goal: Predict impact of disruptions (factory shutdown, port delays)

In [ ]:
@agent struct SupplyChainFirm(GraphAgent)
    # Firm state
    inventory::Float64
    production_capacity::Float64
    current_production::Float64
    
    # Orders and fulfillment
    pending_orders::Dict{Int, Float64}  # orders from downstream firms
    backlog::Float64
    
    # Learned behavior
    safety_stock_target::Float64  # Learned from experience with stockouts
    order_aggressiveness::Float64  # How quickly to order when inventory low
    
    # Type
    firm_type::Symbol  # :supplier, :manufacturer, :distributor
end

function create_supply_chain_twin(n_suppliers=10, n_manufacturers=5, n_distributors=8)
    # Create a directed graph representing supply relationships
    n_total = n_suppliers + n_manufacturers + n_distributors
    graph = SimpleDiGraph(n_total)
    
    # Suppliers (1:n_suppliers) connect to manufacturers
    # Manufacturers connect to distributors
    
    supplier_range = 1:n_suppliers
    manufacturer_range = (n_suppliers+1):(n_suppliers+n_manufacturers)
    distributor_range = (n_suppliers+n_manufacturers+1):n_total
    
    # Each manufacturer gets 2-3 suppliers
    for m in manufacturer_range
        n_suppliers_for_m = rand(2:3)
        suppliers = sample(supplier_range, n_suppliers_for_m, replace=false)
        for s in suppliers
            add_edge!(graph, s, m)
        end
    end
    
    # Each distributor gets 1-2 manufacturers
    for d in distributor_range
        n_mfrs = rand(1:2)
        mfrs = sample(manufacturer_range, n_mfrs, replace=false)
        for m in mfrs
            add_edge!(graph, m, d)
        end
    end
    
    # Initialize model
    model = StandardABM(
        SupplyChainFirm,
        graph;
        agent_step! = supply_chain_step!,
        properties = Dict(
            :total_production => 0.0,
            :total_backlog => 0.0,
            :disruption_active => false,
            :disrupted_firms => Set{Int}()
        )
    )
    
    # Create agents
    for i in 1:n_total
        if i in supplier_range
            firm_type = :supplier
            capacity = rand() * 100 + 50
        elseif i in manufacturer_range
            firm_type = :manufacturer
            capacity = rand() * 80 + 40
        else
            firm_type = :distributor
            capacity = rand() * 60 + 30
        end
        
        add_agent!(
            i,
            model,
            capacity * 0.5,  # inventory (start at 50% of capacity)
            capacity,
            0.0,  # current_production
            Dict{Int, Float64}(),  # pending_orders
            0.0,  # backlog
            capacity * 0.3,  # safety_stock_target (30% of capacity)
            1.0,  # order_aggressiveness
            firm_type
        )
    end
    
    return model
end

### Supply Chain Agent Behavior

- Agents respond to inventory levels
- Learn optimal safety stock from stockout experiences  
- Adapt ordering behavior based on supply reliability

In [ ]:
function supply_chain_step!(agent::SupplyChainFirm, model)
    # Check if disrupted
    is_disrupted = agent.id in model.disrupted_firms
    
    # 1. Production (if not disrupted)
    if !is_disrupted
        # Determine production based on inventory and capacity
        if agent.firm_type == :supplier
            # Suppliers produce at capacity
            production = agent.production_capacity
        else
            # Others need inputs from suppliers
            # Simplified: assume enough inputs if inventory > 0
            if agent.inventory > 0
                production = min(agent.production_capacity, agent.inventory * 0.5)
            else
                production = 0.0
            end
        end
        
        agent.current_production = production
        agent.inventory += production
    else
        agent.current_production = 0.0
    end
    
    # 2. Fulfill orders from downstream (deplete inventory)
    if !isempty(agent.pending_orders)
        total_orders = sum(values(agent.pending_orders))
        
        if agent.inventory >= total_orders
            # Can fulfill all orders
            agent.inventory -= total_orders
            agent.pending_orders = Dict{Int, Float64}()
        else
            # Partial fulfillment, rest goes to backlog
            agent.backlog += (total_orders - agent.inventory)
            agent.inventory = 0.0
            agent.pending_orders = Dict{Int, Float64}()
            
            # Learn: increase safety stock target after stockout
            agent.safety_stock_target *= 1.05
        end
    end
    
    # 3. Place orders to upstream suppliers (if inventory low)
    if agent.firm_type != :supplier
        if agent.inventory < agent.safety_stock_target
            # Order from suppliers
            supplier_ids = inneighbors(model.space, agent.id)
            
            if !isempty(supplier_ids)
                order_quantity = (agent.safety_stock_target - agent.inventory) * agent.order_aggressiveness
                order_per_supplier = order_quantity / length(supplier_ids)
                
                for supplier_id in supplier_ids
                    supplier = model[supplier_id]
                    if !haskey(supplier.pending_orders, agent.id)
                        supplier.pending_orders[agent.id] = 0.0
                    end
                    supplier.pending_orders[agent.id] += order_per_supplier
                end
            end
        end
    end
    
    # 4. Learn: adjust order aggressiveness based on inventory stability
    inventory_ratio = agent.inventory / agent.production_capacity
    if inventory_ratio < 0.2  # Too low
        agent.order_aggressiveness *= 1.02
    elseif inventory_ratio > 0.8  # Too high (over-ordering)
        agent.order_aggressiveness *= 0.98
    end
    agent.order_aggressiveness = clamp(agent.order_aggressiveness, 0.5, 2.0)
end

### Intervention Testing: Simulating Disruptions

- A key use of digital twins: test "what if" scenarios
- What happens if a key supplier is disrupted?
- How far do effects propagate through the network?
- Recall production networks: shocks cascade through interdependencies!

In [ ]:
# Create supply chain twin
sc_model = create_supply_chain_twin(10, 5, 8)

# Run baseline (no disruption)
baseline_model = deepcopy(sc_model)
run!(baseline_model, 50)

baseline_production = [sum(a.current_production for a in allagents(baseline_model))]
println("Baseline total production: ", baseline_production[1])

In [ ]:
# Now test disruption scenario
function simulate_disruption(model, disrupted_firm_id, disruption_start, disruption_duration)
    """
    Simulate a disruption to a specific firm
    """
    production_over_time = Float64[]
    backlog_over_time = Float64[]
    
    for t in 1:100
        # Activate/deactivate disruption
        if t == disruption_start
            push!(model.disrupted_firms, disrupted_firm_id)
            model.disruption_active = true
        elseif t == disruption_start + disruption_duration
            delete!(model.disrupted_firms, disrupted_firm_id)
            model.disruption_active = !isempty(model.disrupted_firms)
        end
        
        # Step simulation
        step!(model)
        
        # Collect metrics
        total_prod = sum(a.current_production for a in allagents(model))
        total_backlog = sum(a.backlog for a in allagents(model))
        
        push!(production_over_time, total_prod)
        push!(backlog_over_time, total_backlog)
    end
    
    return production_over_time, backlog_over_time
end

# Test: disrupt a key supplier
disruption_model = deepcopy(sc_model)
production, backlog = simulate_disruption(disruption_model, 3, 20, 10)

# Visualize impact
plot(
    1:100,
    production,
    xlabel="Time Step",
    ylabel="Total Production",
    label="Production",
    title="Supply Chain Disruption Impact",
    legend=:topright,
    linewidth=2
)

vline!([20], label="Disruption Start", linestyle=:dash, linewidth=2)
vline!([30], label="Disruption End", linestyle=:dash, linewidth=2)

### Exercise 2: Cascading Effects

Using the supply chain digital twin:

1. Identify which firm type (supplier, manufacturer, distributor) causes the largest production drop when disrupted
2. Measure the "recovery time" -- how many steps after disruption ends before production returns to baseline?
3. Test a mitigation strategy: increase safety stock targets for all firms by 50%. Does this reduce the impact?

**Hint**: You'll need to run multiple scenarios and compare total production loss.

In [ ]:
# TODO: Your analysis here

# Test disrupting different firm types
# Measure impact as sum of production losses
# Test mitigation strategy

## Case Study: Urban Mobility Digital Twin

- Let's bring together everything we've learned
- Build a digital twin of urban traffic
- Connect to Braess' Paradox from Week 8!
- Use AI driver agents that learn routing strategies

### The Urban Mobility Challenge

- Cities need to make infrastructure decisions
  - Should we add a new road?
  - How will a new subway line affect traffic?
  - What happens if we close a street for pedestrians?
- Recall Braess' Paradox: adding capacity can *worsen* outcomes!
- Traditional models:
  - Static traffic assignment (assumes fixed routing)
  - Limited ability to model learning and adaptation
- Digital twin approach:
  - AI driver agents learn optimal routes
  - Predict how network changes affect equilibrium
  - Test interventions before implementation

### Traffic Network Setup

- Nodes: Intersections
- Edges: Roads with travel time functions
- Travel time = base_time + congestion_factor * flow
- Driver agents choose routes to minimize travel time

In [ ]:
# Edge data structure for roads
mutable struct Road
    from::Int
    to::Int
    base_time::Float64  # Time with no congestion
    congestion_factor::Float64  # How much each car adds to travel time
    current_flow::Int  # Number of cars currently using this road
    capacity::Int  # Maximum reasonable flow
end

function travel_time(road::Road)
    return road.base_time + road.congestion_factor * road.current_flow
end

# Create the classic Braess network
function create_braess_network(with_shortcut=false)
    # 4 nodes: A (start), B (end), C (top middle), D (bottom middle)
    roads = Road[]
    
    # A -> C: linear congestion (x/100 in original, so base=0, factor=0.01)
    push!(roads, Road(1, 3, 0.0, 0.01, 0, 4000))
    
    # A -> D: constant time
    push!(roads, Road(1, 4, 45.0, 0.0, 0, 4000))
    
    # C -> B: constant time
    push!(roads, Road(3, 2, 45.0, 0.0, 0, 4000))
    
    # D -> B: linear congestion
    push!(roads, Road(4, 2, 0.0, 0.01, 0, 4000))
    
    if with_shortcut
        # C -> D: the "wormhole" - nearly instant
        push!(roads, Road(3, 4, 0.0, 0.0, 0, 4000))
    end
    
    return roads
end

# Helper to find available routes from A to B
function find_routes(roads, start_node, end_node, max_length=3)
    routes = Vector{Vector{Int}}()
    
    # Simple DFS to find all paths
    function dfs(current, target, path, visited)
        if current == target
            push!(routes, copy(path))
            return
        end
        
        if length(path) >= max_length
            return
        end
        
        for road in roads
            if road.from == current && !(road.to in visited)
                push!(path, road.to)
                push!(visited, road.to)
                dfs(road.to, target, path, visited)
                pop!(path)
                pop!(visited)
            end
        end
    end
    
    dfs(start_node, end_node, [start_node], Set([start_node]))
    return routes
end

### AI Driver Agents with Route Learning

- Instead of assuming Nash equilibrium
- Drivers learn through experience
- Use reinforcement learning concepts:
  - Explore: try different routes
  - Exploit: use routes that worked well in the past
  - Adapt: respond to changing traffic patterns

In [ ]:
@agent struct Driver(NoSpaceAgent)
    origin::Int
    destination::Int
    current_route::Vector{Int}  # Sequence of nodes
    route_experiences::Dict{Vector{Int}, Vector{Float64}}  # Route -> travel times
    exploration_rate::Float64  # Probability of trying a new route
    last_travel_time::Float64
end

function create_traffic_twin(n_drivers=4000, with_shortcut=false)
    # Create network
    roads = create_braess_network(with_shortcut)
    
    # Find all possible routes
    available_routes = find_routes(roads, 1, 2)
    
    model = StandardABM(
        Driver;
        agent_step! = driver_step!,
        properties = Dict(
            :roads => roads,
            :available_routes => available_routes,
            :total_travel_time => 0.0,
            :convergence_metric => Float64[]
        )
    )
    
    # Create drivers
    for i in 1:n_drivers
        # Initially assign random routes
        initial_route = rand(available_routes)
        
        add_agent!(
            model,
            1,  # origin (A)
            2,  # destination (B)
            initial_route,
            Dict{Vector{Int}, Vector{Float64}}(),  # No experience yet
            0.2,  # 20% exploration rate
            0.0
        )
    end
    
    return model
end

In [ ]:
function driver_step!(agent::Driver, model)
    # 1. Choose route (explore vs exploit)
    if rand() < agent.exploration_rate || isempty(agent.route_experiences)
        # Explore: try a random route
        agent.current_route = rand(model.available_routes)
    else
        # Exploit: choose best route based on experience
        best_route = agent.current_route
        best_avg_time = Inf
        
        for (route, times) in agent.route_experiences
            if !isempty(times)
                avg_time = mean(times[max(1, end-4):end])  # Use recent experience
                if avg_time < best_avg_time
                    best_avg_time = avg_time
                    best_route = route
                end
            end
        end
        
        agent.current_route = best_route
    end
    
    # 2. Update road flows based on chosen routes
    # (This happens in model step, not agent step)
    
    # 3. Record experience
    if !haskey(agent.route_experiences, agent.current_route)
        agent.route_experiences[agent.current_route] = Float64[]
    end
    push!(agent.route_experiences[agent.current_route], agent.last_travel_time)
    
    # 4. Adapt exploration rate (decrease over time as agents learn)
    total_experiences = sum(length(times) for times in values(agent.route_experiences))
    if total_experiences > 10
        agent.exploration_rate *= 0.99
        agent.exploration_rate = max(0.05, agent.exploration_rate)  # Keep some exploration
    end
end

### Model-Level Dynamics

- After agents choose routes, update road flows
- Calculate travel times based on congestion
- Track convergence to equilibrium

In [ ]:
function model_step!(model)
    # Reset road flows
    for road in model.roads
        road.current_flow = 0
    end
    
    # Count flows based on agent route choices
    for agent in allagents(model)
        route = agent.current_route
        # Each consecutive pair in route is a road
        for i in 1:(length(route)-1)
            from_node = route[i]
            to_node = route[i+1]
            
            # Find corresponding road
            road_idx = findfirst(r -> r.from == from_node && r.to == to_node, model.roads)
            if road_idx !== nothing
                model.roads[road_idx].current_flow += 1
            end
        end
    end
    
    # Calculate travel times for each agent
    total_time = 0.0
    for agent in allagents(model)
        route = agent.current_route
        time = 0.0
        
        for i in 1:(length(route)-1)
            from_node = route[i]
            to_node = route[i+1]
            
            road_idx = findfirst(r -> r.from == from_node && r.to == to_node, model.roads)
            if road_idx !== nothing
                time += travel_time(model.roads[road_idx])
            end
        end
        
        agent.last_travel_time = time
        total_time += time
    end
    
    model.total_travel_time = total_time
    
    # Track variance in travel times as convergence metric
    travel_times = [agent.last_travel_time for agent in allagents(model)]
    push!(model.convergence_metric, std(travel_times))
end

### Experiment: Discovering Braess' Paradox

- Let's see if our learning agents discover the same result!
- Run two scenarios:
  1. Without shortcut (original network)
  2. With shortcut (the "improvement")
- Compare equilibrium outcomes

In [ ]:
# Scenario 1: Without shortcut
println("Running scenario WITHOUT shortcut...")
model_no_shortcut = create_traffic_twin(4000, false)

# Run until convergence
for t in 1:100
    step!(model_no_shortcut, driver_step!, model_step!, 1)
end

avg_time_no_shortcut = model_no_shortcut.total_travel_time / 4000
println("Average travel time WITHOUT shortcut: ", round(avg_time_no_shortcut, digits=2))

In [ ]:
# Scenario 2: With shortcut
println("Running scenario WITH shortcut...")
model_with_shortcut = create_traffic_twin(4000, true)

for t in 1:100
    step!(model_with_shortcut, driver_step!, model_step!, 1)
end

avg_time_with_shortcut = model_with_shortcut.total_travel_time / 4000
println("Average travel time WITH shortcut: ", round(avg_time_with_shortcut, digits=2))

In [ ]:
# Compare
println("\n=== BRAESS' PARADOX ===")
println("Adding the shortcut changed average travel time by: ",
    round(avg_time_with_shortcut - avg_time_no_shortcut, digits=2), " minutes")

if avg_time_with_shortcut > avg_time_no_shortcut
    println("The 'improvement' made things WORSE!")
    println("This is Braess' Paradox in action.")
else
    println("The improvement helped (paradox not observed in this case)")
end

### Visualizing Route Choices

Let's see what routes agents converged to in each scenario.

In [ ]:
function analyze_route_distribution(model)
    route_counts = Dict{Vector{Int}, Int}()
    
    for agent in allagents(model)
        route = agent.current_route
        route_counts[route] = get(route_counts, route, 0) + 1
    end
    
    println("\nRoute Distribution:")
    for (route, count) in sort(collect(route_counts), by=x->x[2], rev=true)
        route_str = join(route, " -> ")
        pct = round(100 * count / nagents(model), digits=1)
        println("  $route_str: $count drivers ($pct%)")
    end
end

println("\n=== WITHOUT SHORTCUT ===")
analyze_route_distribution(model_no_shortcut)

println("\n=== WITH SHORTCUT ===")
analyze_route_distribution(model_with_shortcut)

### Policy Intervention: Can We Fix It?

- We've seen that adding capacity can hurt
- Digital twin benefit: test interventions before implementing
- Let's test a few policies:
  1. Toll on the shortcut
  2. Variable pricing based on congestion
  3. Information provision to drivers

### Exercise 3: Test a Toll Policy

Modify the model to add a toll (fixed time cost) to the C->D shortcut.

**Tasks:**
1. Add a `toll` field to the Road struct
2. Include toll in travel time calculation
3. Test different toll values: 0, 5, 10, 15, 20 minutes
4. Find the toll value that minimizes average travel time
5. Plot average travel time vs toll amount

**Questions:**
- Is there a toll that makes everyone better off than the no-shortcut scenario?
- How does this relate to the concept of a "social planner" from Week 8?

In [ ]:
# TODO: Implement toll policy and test different values

# Hint: You'll need to:
# 1. Modify Road struct (or add toll as a parameter)
# 2. Update travel_time() function
# 3. Run simulations for each toll value
# 4. Collect and compare results

## Real-World Digital Twin Applications

Digital twins are being deployed in production systems worldwide:

### Singapore Virtual City
- Complete 3D digital twin of entire city
- Integrates real-time data from sensors, cameras, IoT devices
- Use cases:
  - Urban planning (test building designs)
  - Emergency response (simulate evacuations)
  - Environmental monitoring (air quality, flooding)

### Amazon Supply Chain Twin
- Models entire fulfillment network
- Predicts demand, inventory needs, delivery times
- Optimizes warehouse placement and routing
- Saved billions in operational costs

### Dutch Water Management
- Digital twin of water systems (dikes, pumps, channels)
- Predicts flooding under different weather scenarios
- Tests emergency response procedures
- Critical for below-sea-level infrastructure

### Siemens Factory Twins
- Virtual model of manufacturing plants
- AI agents represent machines, workers, materials
- Optimizes production schedules
- Predicts maintenance needs before failures occur

## Calibration from Real Data

- A digital twin is only useful if it's accurate
- **Calibration**: The process of adjusting model parameters to match observed data
- Unlike traditional one-time calibration, digital twins calibrate continuously

Let's look at calibration techniques for our traffic twin.

### Calibration Approach

1. **Historical Data**: Use past traffic data to set initial parameters
2. **Real-Time Sync**: Update flows as new data arrives
3. **Parameter Learning**: Adjust behavioral parameters (exploration rate, route preferences)
4. **Validation**: Compare predictions to actual outcomes

**Key Challenge**: Agent heterogeneity
- Not all drivers behave the same way
- Some are more exploratory, some stick to familiar routes
- Some value time more than others

AI agents can capture this heterogeneity by learning individual behavior patterns from data!

### Exercise 4: Calibrating Exploration Rates

Suppose you observe the following from real GPS data:
- 60% of drivers take the same route every day
- 30% occasionally try alternatives when they hear about traffic
- 10% frequently experiment with different routes

**Task**: Modify the driver population to match these behavioral types.

**Steps**:
1. Create three driver types with different exploration rates:
   - Habitual: exploration_rate = 0.05
   - Adaptive: exploration_rate = 0.20
   - Exploratory: exploration_rate = 0.50
2. Set population proportions to match observed data
3. Run simulation and compare route distribution to single-type population
4. Does heterogeneity affect equilibrium travel times?

In [ ]:
# TODO: Implement heterogeneous driver population

# Modify create_traffic_twin to assign different exploration rates
# based on driver type percentages

## Summary: Digital Twins and AI Agents

We've seen how digital twins extend traditional agent-based models:

**Traditional ABMs**:
- Fixed behavioral rules
- One-time calibration
- Focus on understanding emergence

**Digital Twins with AI Agents**:
- Learning, adaptive agents
- Continuous data synchronization
- Operational prediction and optimization
- Real-world deployment for decision support

**Key Insights**:
1. **AI enables realism**: Learning agents capture complex, heterogeneous behavior
2. **Data integration is critical**: Digital twins need continuous calibration
3. **Prediction + prescription**: Not just understanding, but optimizing
4. **Intervention testing**: Safe, low-cost policy experiments

**Connections to Course**:
- Production Networks (Week 5): Economic interdependencies cascade through digital twins
- ABMs (Week 6): Foundation for agent-based digital twins
- Game Theory (Week 8): Strategic behavior, equilibria, Braess' paradox
- Swarms (A3.01): Collective behavior in digital twin systems
- Strategic Agents (A3.02): AI agents with goals and learning

## Looking Ahead

Digital twins represent the cutting edge of AI agent systems:

**Current Frontiers**:
- Multi-scale twins (individual agents + aggregate dynamics)
- Federated learning for privacy-preserving calibration
- Integration with LLMs for natural language interfaces
- Real-time optimization with reinforcement learning

**Challenges**:
- Computational cost (simulating thousands/millions of agents)
- Data availability and quality
- Validation and trust (how do we know predictions are reliable?)
- Ethical considerations (surveillance, manipulation)

**Next Week (A4)**: We'll explore production deployment of AI agent systems and future directions in agentic AI.

**Your Projects**: Consider how digital twins could enhance your proposed AI agent system. Could you:
- Sync with real data sources?
- Enable what-if scenario testing?
- Predict system behavior before deployment?
- Optimize policies or interventions?

## Additional Exercises

### Exercise 5: Multi-Modal Transportation

Extend the urban mobility twin to include multiple transportation modes:
- Private car (current model)
- Public transit (bus with fixed route, capacity constraints)
- Bike/scooter (weather-dependent, slower but no congestion)

Agents choose mode based on:
- Time cost
- Monetary cost
- Personal preferences (learned from data)

**Test**: What happens if you add a new bus line? Does it reduce car traffic?

### Exercise 6: Social Network Digital Twin

Build a digital twin of a social network platform:
- Agents: Users with interests, attention budgets
- Content: Posts with quality, relevance, virality
- Platform: Recommendation algorithm

**Goal**: Predict engagement and optimize content recommendation.

**Calibration**: Use real engagement data (likes, shares, comments) to learn:
- User interest profiles
- Content quality metrics
- Virality factors

**Intervention**: Test algorithm changes before deploying to real users.

### Exercise 7: Economic Shock Propagation

Revisit the production network from Week 5:
1. Build a digital twin with AI firm agents
2. Calibrate using real input-output data
3. Simulate sector-specific productivity shock
4. Measure how shock propagates through supply chains

**Compare**:
- Analytical Leontief model (Week 5): $\Delta x = L \Delta d$
- Digital twin with learning agents: How do results differ?

**Extension**: Can firms learn to diversify suppliers to reduce shock exposure?